# Clearing quarantined observations

A check holds back an observation it doubts. The observation stays out of training until you
clear it here. Each decision is either **confirmed** (the value is real) or **rejected** (the
value is wrong), and each needs a reason.

Verify a surprising value against the world before you reject it. Read the report, compare the
park count, and check the weather that day. An unusual count may be real.

Reasons stay in the database. The dashboard shows the decision, never the reason. Clear this
notebook's outputs before you commit it, because the outputs can show report values.

## Open the database

The daily run and this notebook cannot write to the database at the same time. Close the
connection at the end, or the next scheduled run will fail to get its lock.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # run from notebooks/ without installing the package

from roll_call.quality import clearing
from roll_call.storage import db

con = db.connect()

## List the open items

Each row is one quarantined observation with no decision yet. `obs_id` names the table and the
key, such as `blue_spring_counts_daily:2026-01-05`. `check_name` says which check doubted it,
and `value` is what it saw.

In [ ]:
import pandas as pd

items = pd.DataFrame(clearing.open_items(con))
items

## Record a decision

Copy an `obs_id` from the table above. Set `decision` to `"confirmed"` or `"rejected"`, and
write a reason someone else could check later. The call refuses an unknown decision, an empty
reason, and a second decision on the same observation. Pass `replace=True` to change a decision
on purpose.

In [ ]:
obs_id = ""    # for example "blue_spring_counts_daily:2026-01-05"
decision = ""  # "confirmed" or "rejected"
reason = ""    # what you checked, and what it showed

clearing.decide(con, obs_id, decision, reason)

## Check what is left, then close

The item you decided should be gone from this list.

In [ ]:
pd.DataFrame(clearing.open_items(con))

In [ ]:
con.close()